In [ ]:
import fasttext.util
import numpy as np
import pandas as pd
import re
import os
from scipy import stats
from scipy.stats import skew, kurtosis
from collections import Counter
from nltk.corpus import stopwords
import nltk
import spacy
import matplotlib.pyplot as plt
import plotly.graph_objects as go

ft = fasttext.load_model(r"Data/Models/cc.de.300.bin") #fasttext model path
print("Model loaded successfully.")

Model loaded successfully.


In [ ]:
#path of a single file
file_path = r"Data/Corpus/Altenberg_Die_Natur.txt"

try:
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
except Exception as e:
    print(f"Error reading: {e}")

nlp = spacy.load("de_core_news_md")

nltk.download('stopwords', quiet=True)
german_stop_words = set(stopwords.words('german'))

key_distinction_1 = ['Natur']
key_distinction_2 = ['Kultur']

#key_distinction_1 = ['forms_of_representation_for_key_distinction_1']
#key_distinction_2 = ['forms_of_representation_for_key_distinction_2']

"""
#Example

#for adjectives
key_distinction_1 = ['naturverbunden']
key_distinction_2 = ['zivilisiert']

#average of synonyms
key_distinction_1 = ['naturverbunden', 'naturbelassen', 'rein', 'unverfälscht', 'lebendig', 'natürlich']
key_distinction_2 = ['zivilisiert', 'kultiviert', 'künstlich', 'artifiziell', 'hochentwickelt', 'gebildet', 'gekünstelt']

#nouns
key_distinction_1 = ['Natur']
key_distinction_2 = ['Kultur']
"""

print("Checking OOV words for the selected synonyms:")
for w in key_distinction_1 + key_distinction_2:
    if ft.get_word_id(w) == -1:
        print(f"'{w}' is OOV!")

#find axis vector and normalize it
def get_axis_vector():
    key_distinction_1_vecs = [ft.get_word_vector(w) for w in key_distinction_1]
    key_distinction_2_vecs = [ft.get_word_vector(w) for w in key_distinction_2]
    key_distinction_1_mean = np.mean(key_distinction_1_vecs, axis=0)
    key_distinction_2_mean = np.mean(key_distinction_2_vecs, axis=0)
    axis = key_distinction_2_mean - key_distinction_1_mean
    return axis / np.linalg.norm(axis)

#find cosine similarities
def cosine_to_axis(word, axis_vec):
    v = ft.get_word_vector(word)
    norm = np.linalg.norm(v)
    if norm == 0:
        return 0.0
    v_norm = v / norm
    return float(np.dot(v_norm, axis_vec))

#handle empty lists to avoid errors
def summary_stats(scores):
    if not scores: 
        return {k: 0.0 for k in ["min", "max", "diff", "mean", "median", "std", "skew", "kurt"]}
        
    scores = np.array(scores)
    return {
        "min":    float(np.min(scores)),
        "max":    float(np.max(scores)),
        "diff":   float(np.max(scores)-np.min(scores)),
        "mean":   float(np.mean(scores)),
        "median": float(np.median(scores)),
        "std":    float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,
        "skew":   float(stats.skew(scores, bias=False)) if len(scores) > 1 else 0.0,
        "kurt":   float(stats.kurtosis(scores, fisher=True, bias=False)) if len(scores) > 1 else 0.0,
    }

def calculate_file_bias(text, axis_vec, seed_words):
    words = re.sub(r"[^\w\s]", "", text.lower()).split()
    
    unique_text_words = set(words)
    
    #Keeps only words that are stopwords
    neutral_candidates = unique_text_words.intersection(german_stop_words)
    
    scores = []
    for w in neutral_candidates:
        # Checks for seed words
        if w in seed_words: continue
        
        #Checks if the word exists in the FastText model
        if ft.get_word_id(w) != -1:
            scores.append(cosine_to_axis(w, axis_vec))

    if not scores: return 0.0
    return np.mean(scores)

#function for the pipline
def process_text_pipeline(text):
    #Runs SpaCy on RAW text, Filters NOUN/ADJ, Lowercases and cleans punctuation, Removes Stopwords, Checks if word exists in FastText Vocab
    doc = nlp(text)
    valid_words = []
    total_count = 0
    oov = 0
    
    for token in doc:
        #POS Tagging
        if token.pos_ in {"NOUN", "ADJ"}:
            #Lowercase & Clean
            w_clean = re.sub(r"[^\w\s]", "", token.text.lower())
            #Check Stopwords & Empty strings
            if w_clean and w_clean not in german_stop_words:
                total_count +=1 #keeps all the words
                if ft.get_word_id(w_clean) != -1: #Check FastText Vocabulary
                    valid_words.append(w_clean)
                else:
                    oov +=1

    if total_count > 0:
        pct = (oov / total_count) * 100
        print(f"Vocabulary Info: {total_count} total words, {oov} OOV ({pct:.1f}% dropped)")
    else:
        print(f"Vocabulary Info: No candidates found.")
    return valid_words

axis_vec = get_axis_vector()
print(f"Analyzing text, word length is: {len(text)}")

all_seed_words = set(key_distinction_1 + key_distinction_2)
file_bias = calculate_file_bias(text, axis_vec, all_seed_words)

print(f"Document bias: {file_bias:.4f}")

content_words = process_text_pipeline(text)

if content_words:
    unique_words = sorted(set(content_words))

    temp_scores = []

    for w in unique_words:
        raw = cosine_to_axis(w, axis_vec)
    
        #bias correction
        corrected = raw - file_bias
    
        temp_scores.append((w, corrected))

    plot_entries = temp_scores

    if plot_entries:
        freqs = Counter(content_words)
        plot_words  = [x[0] for x in plot_entries]
        plot_scores = [x[1] for x in plot_entries]
        plot_counts = [freqs[w] for w in plot_words]

        scores_freq = []
        for s, c in zip(plot_scores, plot_counts):
            scores_freq.extend([s] * c)
            
        u_st = summary_stats(plot_scores)
        f_st = summary_stats(scores_freq)
        
        print(f"For unique Words:")
        print(f"  min    : {u_st['min']:.4f}")
        print(f"  max    : {u_st['max']:.4f}")
        print(f"  diff   : {u_st['diff']:.4f}")
        print(f"  mean   : {u_st['mean']:.4f}")
        print(f"  median : {u_st['median']:.4f}")
        print(f"  std    : {u_st['std']:.4f}")
        print(f"  skew   : {u_st['skew']:.4f}")
        print(f"  kurt   : {u_st['kurt']:.4f}")
    
        print("Considering the frequencey:")
        print(f"  min    : {f_st['min']:.4f}")
        print(f"  max    : {f_st['max']:.4f}")
        print(f"  diff   : {f_st['diff']:.4f}")
        print(f"  mean   : {f_st['mean']:.4f}")
        print(f"  median : {f_st['median']:.4f}")
        print(f"  std    : {f_st['std']:.4f}")
        print(f"  skew   : {f_st['skew']:.4f}")
        print(f"  kurt   : {f_st['kurt']:.4f}")
        
        #Apply Jitters in the y-axis        
        n = len(plot_scores)
        ys = np.linspace(-1.5, 1.5, n)
        
        # Randomize jitter assignment
        perm = np.random.permutation(n)
        ys = ys[perm]
        sizes = [c * 30 for c in plot_counts] #change for bubble size
    
        plt.figure(figsize=(12, 10))
        plt.scatter(plot_scores, ys, s=sizes, alpha=0.5, c='skyblue', edgecolors='grey')
    
        for x, y, lab, count in zip(plot_scores, ys, plot_words, plot_counts):
                plt.text(x, y, lab, fontsize=9, alpha=0.8)
    
        plt.axvline(0, color="black", linestyle="--", linewidth=1)
        
        plt.yticks([])
        ax = plt.gca()

        # Separator at x=0
        ax.text(
            0, -0.10, "|",
            transform=ax.get_xaxis_transform(),
            ha="center",
        )
        
        #Place x-axis titles
        ax.annotate(
            "Natural",
            xy=(0, -0.10),
            xycoords=ax.get_xaxis_transform(),
            xytext=(-40, 0),      
            textcoords="offset points",
            ha="right",
            va="top",
            fontsize=12
        )
        
        ax.annotate(
            "Cultural",
            xy=(0, -0.10),
            xycoords=ax.get_xaxis_transform(),
            xytext=(40, 0),      
            textcoords="offset points",
            ha="left",
            va="top",
            fontsize=12
        )
        
        plt.tight_layout()
        plt.savefig(
        "scatter plot (Natur_Kultur_Altenberg_Die_Natur).png",
        dpi=300,              # high-resolution image
        bbox_inches="tight"   # trims extra whitespace
        )
        plt.show()
else:
    print("No valid content words found in paragraph.")